# Clothing Brand Big Data — Million-Row Collector (YouTube API)
**Big Data Analytics weekly project** · One combined CSV for product/price clustering.

- Collect real accessible public top-level comments, video metadata and **unverified** price mentions.
- Store rows incrementally in **SQLite** with a unique comment ID and resumable video checkpoint.
- Export **one CSV** with one comment per row. **Millions of records are a target, not a guarantee**: relevant YouTube videos, API quota and comments availability constrain actual volume.
- YouTube does **not** provide verified selling prices; `verified_price_pkr` remains blank until you join an authorized product catalog. `product_candidate` is the video title, **not** a verified product name.
- For price-based clustering, filter for `product_match_status == verified` after product verification. Search brand matches are not proof of an official channel.

**Safety/privacy:** Use only permitted API access, do not publish commenter identifiers, follow YouTube API data retention and deletion policies, and don't share your API key.

## 1. Install dependencies

In [ ]:
!pip -q install google-api-python-client pandas tqdm

## 2. Settings, API key and persistent storage

For Colab continuity, optional Google Drive mounting stores the SQLite database there. Drive-backed SQLite can be slower; do not run multiple collectors against the same database simultaneously. You can keep data locally for a single session by setting `USE_GOOGLE_DRIVE = False`.

In [ ]:
import os, re, time, json, sqlite3, csv
from pathlib import Path
from getpass import getpass
from datetime import datetime, timezone
from urllib.parse import urlparse
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from tqdm.auto import tqdm
import pandas as pd

USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/clothing_big_data')
else:
    OUT = Path('/content/clothing_big_data')
OUT.mkdir(parents=True, exist_ok=True)
DB_FILE = OUT / 'youtube_collection.sqlite3'
CSV_FILE = OUT / 'youtube_clothing_ONE_FILE.csv'

API_KEY = getpass('Paste ONLY your YouTube Data API key: ').strip()
if not API_KEY.startswith('AIza'):
    raise ValueError('Google API key should begin with AIza. Do not paste Python code here.')
youtube = build('youtube', 'v3', developerKey=API_KEY, cache_discovery=False)

BRANDS = ['Khaadi','Sapphire','Gul Ahmed','Alkaram Studio','Nishat Linen','Limelight','Beechtree','Bonanza Satrangi','J.','Ethnic']
VIDEOS_PER_BRAND = 100  # Increase gradually; each search page consumes quota
MAX_COMMENTS_PER_VIDEO = 3000  # Actual accessible comments may be much lower
PAUSE_SECONDS = 0.15
print('Database:', DB_FILE)
print('Output CSV:', CSV_FILE)

## 3. API connection test — run before collection

In [ ]:
try:
    r = youtube.videos().list(part='snippet', id='dQw4w9WgXcQ').execute()
    print('API connection succeeded. Video records:', len(r.get('items', [])))
except HttpError as e:
    raise RuntimeError(f'API request failed (HTTP {e.resp.status}): {e}')

## 4. Product/category/price candidates
Price regex finds price **mentions** in titles/descriptions, not verified prices. Multiple mentions are left ambiguous.

In [ ]:
CATEGORY_KEYWORDS = {
    '3 Piece Suit': ['3 piece','3-piece','3pc','3 pcs'],
    '2 Piece Suit': ['2 piece','2-piece','2pc','2 pcs'],
    'Kurti': ['kurti','kurta'], 'Lawn': ['lawn'],
    'Shirt': ['shirt'], 'Abaya': ['abaya'], 'Dress': ['dress','frock']
}
PRICE_RE = re.compile(r'(?<!\w)(?:PKR|Rs\.?|₨)\s*([0-9][0-9,]*(?:\.[0-9]{1,2})?)\b|\b([0-9][0-9,]*(?:\.[0-9]{1,2})?)\s*(?:PKR|Rs\.?)\b', re.I)
URL_RE = re.compile(r'https?://[^\s<>"\']+',re.I)

def product_candidate(title, desc):
    combined = f'{title or ""}\n{desc or ""}'
    low = combined.lower()
    cat = next((k for k,v in CATEGORY_KEYWORDS.items() if any(x in low for x in v)), 'Unknown')
    vals = []
    for x,y in PRICE_RE.findall(combined):
        try: vals.append(float((x or y).replace(',','')))
        except ValueError: pass
    links = [s.rstrip('.,);') for s in URL_RE.findall(desc or '')]
    return dict(product_candidate=title,product_category_candidate=cat,
                price_mention_pkr=vals[0] if len(vals)==1 else None,
                price_mentions_count=len(vals), price_source='video_text_unverified' if vals else 'not_found',
                verified_price_pkr=None, product_match_status='unverified',
                product_url_candidate=next((u for u in links if any(z in urlparse(u).netloc.lower() for z in ['khaadi','sapphire','gulahmed','alkaram','daraz'])), None))
print(product_candidate('3 Piece Suit Rs. 4,500','Shop link https://example.com'))

## 5. SQLite store (deduplicated, resumable)
One record per YouTube top-level comment. Completed videos are skipped in later runs. API results should be refreshed periodically in accordance with YouTube data policies.

In [ ]:
con = sqlite3.connect(str(DB_FILE), timeout=60)
con.execute('PRAGMA busy_timeout=60000')
con.execute('CREATE TABLE IF NOT EXISTS videos (video_id TEXT PRIMARY KEY, data_json TEXT NOT NULL)')
con.execute('CREATE TABLE IF NOT EXISTS comments (comment_id TEXT PRIMARY KEY, video_id TEXT NOT NULL, data_json TEXT NOT NULL)')
con.execute('CREATE INDEX IF NOT EXISTS idx_comments_video ON comments(video_id)')
con.execute('CREATE TABLE IF NOT EXISTS completed (video_id TEXT PRIMARY KEY, completed_at TEXT NOT NULL)')
con.commit()
print('Previously saved comments:', con.execute('SELECT COUNT(*) FROM comments').fetchone()[0])

## 6. Paginated video discovery (50 per page) and metadata (50 per batch)

In [ ]:
def search_ids(brand, limit):
    ids=[]; seen=set(); used_pages=set(); token=None
    while len(ids) < limit:
        args = dict(part='snippet',q=f'{brand} Pakistan clothing collection',type='video',maxResults=min(50,limit-len(ids)),order='relevance')
        if token: args['pageToken']=token
        data = youtube.search().list(**args).execute()
        items = data.get('items',[])
        if not items: break
        for item in items:
            vid=item.get('id',{}).get('videoId')
            if vid and vid not in seen: ids.append(vid); seen.add(vid)
        nxt=data.get('nextPageToken')
        if not nxt or nxt in used_pages: break
        used_pages.add(nxt);token=nxt
        time.sleep(PAUSE_SECONDS)
    return ids[:limit]

def store_video_details(ids,brand):
    added=0
    for offset in range(0,len(ids),50):
        r=youtube.videos().list(part='snippet,statistics',id=','.join(ids[offset:offset+50])).execute()
        for item in r.get('items',[]):
            sn=item.get('snippet',{});st=item.get('statistics',{});vid=item['id']
            title=sn.get('title','');desc=sn.get('description','')
            row=dict(platform='YouTube',brand_search_term=brand,video_id=vid,
                     video_url=f'https://www.youtube.com/watch?v={vid}',video_title=title,
                     video_description=desc,channel_title=sn.get('channelTitle'),channel_id=sn.get('channelId'),
                     video_published_at=sn.get('publishedAt'),video_views=int(st['viewCount']) if 'viewCount' in st else None,
                     video_likes=int(st['likeCount']) if 'likeCount' in st else None,
                     reported_comment_count=int(st['commentCount']) if 'commentCount' in st else None,
                     **product_candidate(title,desc))
            con.execute('INSERT OR IGNORE INTO videos(video_id,data_json) VALUES (?,?)',(vid,json.dumps(row,ensure_ascii=False)))
            added+=1
        con.commit();time.sleep(PAUSE_SECONDS)
    return added

for brand in tqdm(BRANDS,desc='Searching brands'):
    try:
        ids=search_ids(brand,VIDEOS_PER_BRAND)
        count=store_video_details(ids,brand)
        print(f'{brand}: {count} video metadata records retrieved')
    except HttpError as e:
        print(f'HTTP {e.resp.status} on {brand}: {str(e)[:220]}')
        if e.resp.status in (403,429):
            print('Stopping on access/quota error. Re-run in a permitted later session.')
            break
print('Unique stored videos:',con.execute('SELECT COUNT(*) FROM videos').fetchone()[0])

## 7. Download comments and save after **each page**
Handles quota interruptions without losing earlier pages. Incomplete videos restart from page 1 on later runs; existing comment IDs are deduplicated by SQLite. Some videos have comments disabled. Video becomes completed only after the final page or the configured cap.

In [ ]:
def collect_comments_for_video(video_id,limit=3000):
    if con.execute('SELECT 1 FROM completed WHERE video_id=?',(video_id,)).fetchone():
        return 0
    token=None; received=0; fresh=0
    while received < limit:
        args=dict(part='snippet',videoId=video_id,maxResults=min(100,limit-received),textFormat='plainText',order='time')
        if token:args['pageToken']=token
        try:
            data=youtube.commentThreads().list(**args).execute()
        except HttpError as e:
            # Comments disabled or inaccessible: skip; do not mark as completed, so later attempts are possible
            if e.resp.status in (403,404) and any(x in str(e).lower() for x in ['commentsdisabled','videonotfound','forbidden']):
                print('Unavailable comments:',video_id)
                return fresh
            raise
        items=data.get('items',[])
        for item in items:
            c=item.get('snippet',{}).get('topLevelComment',{})
            sn=c.get('snippet',{}); message=sn.get('textDisplay','').strip()
            if not message:continue
            row=dict(comment_id=c.get('id'),video_id=video_id,comment_text=message,
                     comment_likes=sn.get('likeCount',0),comment_published_at=sn.get('publishedAt'),
                     comment_updated_at=sn.get('updatedAt'))
            cur=con.execute('INSERT OR IGNORE INTO comments(comment_id,video_id,data_json) VALUES (?,?,?)',
                            (row['comment_id'],video_id,json.dumps(row,ensure_ascii=False)))
            fresh+=cur.rowcount
        con.commit()
        received+=len(items)
        token=data.get('nextPageToken')
        if not token or not items:break
        time.sleep(PAUSE_SECONDS)
    con.execute('INSERT OR REPLACE INTO completed VALUES (?,?)', (video_id,datetime.now(timezone.utc).isoformat()))
    con.commit()
    return fresh

ids=[r[0] for r in con.execute('SELECT video_id FROM videos ORDER BY video_id')]
for vid in tqdm(ids,desc='Collecting comments'):
    try:
        collect_comments_for_video(vid,MAX_COMMENTS_PER_VIDEO)
    except HttpError as e:
        print('Collection interrupted at video:',vid,'HTTP:',e.resp.status)
        print('Progress saved. Fix the access/quota error then rerun Cell 7.')
        break
print('Unique saved comments:',con.execute('SELECT COUNT(*) FROM comments').fetchone()[0])
print('Completed videos:',con.execute('SELECT COUNT(*) FROM completed').fetchone()[0])

## 8. Export **ONE** flat CSV, streaming rows to disk
Every row represents one comment with video/product candidate fields. No giant pandas merge or RAM-intensive DataFrame needed. CSV may be large (GBs).

In [ ]:
COLUMNS = [
 'platform','brand_search_term','video_id','video_url','video_title','channel_title','channel_id','video_published_at',
 'video_views','video_likes','reported_comment_count',
 'product_candidate','product_category_candidate','product_url_candidate','price_mention_pkr','price_mentions_count',
 'price_source','verified_price_pkr','product_match_status',
 'comment_id','comment_text','comment_likes','comment_published_at','comment_updated_at',
 'sentiment','price_complaint','collected_at'
]

def export_single_csv():
    count=0
    with open(CSV_FILE,'w',newline='',encoding='utf-8-sig') as f:
        w=csv.DictWriter(f,fieldnames=COLUMNS,extrasaction='ignore')
        w.writeheader()
        cur=con.execute('SELECT c.data_json,v.data_json FROM comments c JOIN videos v ON v.video_id=c.video_id ORDER BY c.comment_id')
        now=datetime.now(timezone.utc).isoformat()
        for comment_json,video_json in cur:
            item={**json.loads(video_json),**json.loads(comment_json)}
            item.update(sentiment='',price_complaint='',collected_at=now)
            w.writerow(item)
            count+=1
    return count

rows=export_single_csv()
print('Exported rows:',rows)
print('Single CSV:',CSV_FILE)
print('File size MB:',round(CSV_FILE.stat().st_size/1024**2,2))

## 9. Check CSV with streaming / chunks
Avoid `pd.read_csv()` loading millions of rows at once.

In [ ]:
print(pd.read_csv(CSV_FILE,nrows=5)[['brand_search_term','video_title','comment_text','product_category_candidate','price_mention_pkr','verified_price_pkr']])
counts={}
for chunk in pd.read_csv(CSV_FILE,usecols=['brand_search_term'],chunksize=100_000):
    for brand,num in chunk['brand_search_term'].value_counts().items():
        counts[brand]=counts.get(brand,0)+int(num)
print('Comments per brand search term:',counts)

## 10. Using this for clustering
- **Do not** use `price_mention_pkr` as actual price; it can be from multiple products, discounts or unrelated text. First verify SKU/product and selling price via permitted brand catalog, update `verified_price_pkr` and set `product_match_status=verified`.
- `brand_search_term` is the search query label, not independently verified brand ownership.
- Fill `sentiment` with a properly evaluated classifier after manual ground-truth labeling.
- Cluster **products** using aggregated product features (verified price, sentiment proportions, price complaint fraction, comment volumes), not millions of repeated comment rows as if they were independent products.
- Actual customer age/location is not recoverable reliably from public YouTube comments.
- To expand from thousands toward millions, get more relevant authorized video IDs, repeat allowed sessions within quota, and consider partitioned Parquet/PySpark for processing even though this notebook exports the requested single CSV.
- CSV is a deliverable, not an efficient storage engine for very large interactive analytics; SQLite is the resumable source of truth.